### MY470 Computer Programming
# Same Votes, Different Winners: Electoral Systems
### Week 7 Lab

<div style="box-sizing: border-box; margin-top: 28px; max-width: 684px; font-family: system-ui; padding: 14px 22px; background-color: #FFFFFF; border-left: 8px solid #4f6e51; border-radius: 8px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1); color: #212121;">
<div style="line-height: 1.3; font-weight: bold;">Student Notebook</div>
<div style="margin-top: 0.4em; line-height: 1.6; font-size: 0.8em;">
Date: [Add today's date here]<br>
Name: [Add your name here]<br>
Purpose: Practice inheritance, overriding, class hierarchies, protecting data, and generators
</div>
</div>

## Background

Who wins an election depends not only on the votes, but also on the rules that count them. Political scientists have studied this for a long time, and Kenneth Arrow showed that no ranked voting system can satisfy a short list of reasonable fairness conditions all at once.

Today, we'll write programs to count the same ballots under different electoral systems to answer the question: **can the same votes produce different winners?**

<div class="section-label">Part 1</div>

# Counting Votes

## The Ballots

Each voter ranks the parties from best to worst. These 100 ballots are invented.

In [1]:
BALLOTS = ([['Amber', 'Crimson', 'Blue'] for i in range(40)]
           + [['Blue', 'Crimson', 'Amber'] for i in range(35)]
           + [['Crimson', 'Blue', 'Amber'] for i in range(25)])

print(len(BALLOTS), BALLOTS[0])

100 ['Amber', 'Crimson', 'Blue']


## The Class `Election`

Each system is a subclass that overrides `winner()`. The class variable `system` gives its name.

In [2]:
class Election(object):
    """An election. Subclasses define how the winner is found."""
    system = 'General election'

    def __init__(self, ballots):
        self.ballots = ballots

    def get_ballots(self):
        return self.ballots

    def count_first_choices(self):
        """Returns a dictionary: party -> number of first choices."""
        counts = {}
        for ballot in self.ballots:
            party = ballot[0]
            if party not in counts:
                counts[party] = 0
            counts[party] += 1
        return counts

    def winner(self):
        """Returns the winning party. Subclasses override this."""
        return None

    def __str__(self):
        return self.system + ': ' + str(self.winner())

## Two Kinds of Elections

Some systems use only the first choices. Others use the full rankings: they inherit from `RankedElection`.

In [ ]:
class RankedElection(Election):
    """An election that uses the voters' full rankings."""
    pass

## Exercise 1

**Plurality** (as in UK general elections): the party with the most first choices wins.

Write the class `Plurality`, a subclass of `Election`, by replacing `pass` in the cell below:

1. Set the class variable `system` to `'Plurality'`.
2. Override `winner()`. It takes no arguments apart from `self` and **returns** the name of the winning party as a string, for example `'Amber'`.
3. Inside `winner()`, get the number of first choices by calling the inherited method `count_first_choices()`. Do not loop over the ballots again.

Do not write `__str__()`: `print()` uses the inherited method, which calls your `winner()`. If parties tie, return any one of them.

In [ ]:
class Plurality(Election):
    pass

# Test
print(Plurality(BALLOTS))

# Expected output:
# Plurality: Amber

## Exercise 2

**Runoff:** only the two parties with the most first choices remain. Each ballot then counts for whichever of the two it ranks higher.

Write the class `Runoff`, a subclass of `RankedElection`:

1. Set the class variable `system` to `'Runoff'`.
2. Override `winner()`. It **returns** the name of the winning party as a string, and works in three steps:
    * **Find the top two:** call `count_first_choices()` and take the two parties with the highest counts.
    * **Count the second round:** for each ballot in `self.ballots`, give one vote to whichever of the two parties appears first on it.
    * **Compare:** return the party with more votes. If they tie, return either one.

**Hint:** if `counts` is a dictionary, `sorted(counts, key=counts.get, reverse=True)` returns a list of its keys, ordered from the highest value to the lowest.

In [ ]:
class Runoff(RankedElection):
    pass

# Test
print(Runoff(BALLOTS))

# Expected output:
# Runoff: Blue

## Exercise 3

**Borda count:** with three parties, a first place on a ballot earns 2 points, a second place 1 point, and a last place 0 points. The party with the most points wins.

Write the class `Borda`, a subclass of `RankedElection`:

1. Set the class variable `system` to `'Borda count'`.
2. Override `winner()`. It adds up each party's points over all ballots and **returns** the name of the party with the most points as a string. If parties tie, return any one of them.
3. Your code must work for any number of parties, so do not assume there are three. On a ballot with `n` parties, the party at index `i` earns `n - 1 - i` points.

The test below therefore also uses ballots with four parties.

In [ ]:
class Borda(RankedElection):
    pass

# Test
print(Borda(BALLOTS))

four_parties = ([['A', 'B', 'C', 'D'] for i in range(3)]
                + [['D', 'B', 'C', 'A'] for i in range(2)])
print(Borda(four_parties))

# Expected output:
# Borda count: Crimson
# Borda count: B

<div class="section-label">Part 2</div>

# Comparing Systems

## Three Systems, One Set of Ballots

Run this cell. `print()` does not need to know which kind of election it gets.

In [ ]:
elections = [Plurality(BALLOTS), Runoff(BALLOTS), Borda(BALLOTS)]
for election in elections:
    print(election, isinstance(election, RankedElection))

## Discussion

* The same 100 ballots produce three different winners. Which party *should* win, and why?
* The loop calls `print(election)` without knowing the type of `election`. Which methods run for a `Borda` object?
* Suppose a new subclass's `winner()` returned a list of parties when there is a tie. What might that break, according to the substitution principle?

<div class="section-label">Part 3</div>

# Stuffing the Ballot Box

## A Small Fraud

Run this: what happens to the result?

In [ ]:
ballots = [ballot for ballot in BALLOTS]
election = Plurality(ballots)
print(election)

extra = [['Blue', 'Crimson', 'Amber'] for i in range(10)]
election.get_ballots().extend(extra)
print(election)

ballots.extend([['Crimson', 'Blue', 'Amber'] for i in range(40)])
print(election)

## Exercise 4

The fraud above used **two** holes:

* **`get_ballots()`** returns the election's own list, so extending the returned list changes the election.
* **`__init__()`** stores the list it receives, so extending that list later changes the election.

Write the class `SecurePlurality`, a subclass of `Plurality`, that overrides both methods:

1. `__init__(self, ballots)`: make a copy of `ballots` and pass the copy to `super().__init__()`.
2. `get_ballots(self)`: call `super().get_ballots()` and return a copy of the result.

A copy of the outer list is enough, for example with `[:]`. Do not override `winner()` or set `system`: both are inherited.

In [ ]:
class SecurePlurality(Plurality):
    pass

# Test
ballots = [ballot for ballot in BALLOTS]
election = SecurePlurality(ballots)
extra = [['Blue', 'Crimson', 'Amber'] for i in range(10)]
election.get_ballots().extend(extra)
ballots.extend([['Crimson', 'Blue', 'Amber'] for i in range(40)])
print(election, len(election.get_ballots()))

# Expected output:
# Plurality: Amber 100

## Discussion

* Each ballot is itself a list. Can someone still change a ballot that is already in the election? How? What would you need to do to prevent this?
* Can outside code still reach `election.ballots` directly? Would naming it `self._ballots` change that?

<div class="section-label">Part 4</div>

# Auditing the Count

## Exercise 5

An audit goes through the ballots one by one. With millions of ballots, we do not want to build a new list for this.

Write the class `AuditedElection`, a subclass of `Election`, with one new method:

* `ballots_for(self, party)` takes the name of a party as a string and is a **generator**. It goes through `self.ballots` and uses `yield` to produce, one at a time, each ballot whose first choice is `party`.
* Each item it produces is a whole ballot, that is, a list such as `['Crimson', 'Blue', 'Amber']`.
* The method must not build or return a list.

You do not need to override `winner()` here.

In [ ]:
class AuditedElection(Election):
    pass

# Test
audit = AuditedElection(BALLOTS)
count = 0
for ballot in audit.ballots_for('Crimson'):
    count += 1
print(count)

# Expected output:
# 25